# LeetCode 1133 — Largest Unique Number bằng Python

**Link đề:** https://leetcode.com/problems/largest-unique-number/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def largestUniqueNumber(self, nums: List[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu đúng "số **xuất hiện đúng một lần**" (unique) khác với "số **phân biệt**" (distinct, sau khi bỏ trùng).
2. Thành thạo **đếm tần suất** bằng `dict`, `collections.Counter` và mảng đếm.
3. Đi từ Brute Force `O(n²)` → sắp xếp `O(n log n)` → đếm `O(n)` → mảng đếm duyệt ngược `O(n + V)`.
4. Trace, chứng minh tính đúng, phân tích Time/Space, kiểm thử đối chiếu.
5. Tự giải biến thể: số unique nhỏ nhất, tổng các số unique (#1748), số unique đầu tiên trong luồng.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — số lớn nhất chỉ xuất hiện một lần

Cho mảng số nguyên `nums`, trả về **số nguyên lớn nhất xuất hiện đúng một lần**. Nếu không có số nào xuất hiện đúng một lần, trả về `-1`.

| nums | Output | Giải thích |
|---|---:|---|
| `[5,7,3,9,4,9,8,3,1]` | 8 | 9 xuất hiện 2 lần (loại), 8 xuất hiện 1 lần và là lớn nhất còn lại |
| `[9,9,8,8]` | -1 | mọi số đều lặp |

**Constraints (theo đề gốc):** `1 <= nums.length <= 2000`, `0 <= nums[i] <= 1000`.

In [1]:
nums = [5, 7, 3, 9, 4, 9, 8, 3, 1]
print("max(nums) =", max(nums), "nhưng 9 xuất hiện", nums.count(9), "lần -> không hợp lệ")
print("8 xuất hiện", nums.count(8), "lần -> hợp lệ")

max(nums) = 9 nhưng 9 xuất hiện 2 lần -> không hợp lệ
8 xuất hiện 1 lần -> hợp lệ


## 2. Nền tảng: "unique" khác "distinct"

- **Distinct** (phân biệt): bỏ trùng, mỗi giá trị giữ lại một bản → `set(nums)`. Với ví dụ 1: `{1,3,4,5,7,8,9}` — **vẫn có 9**.
- **Unique** (duy nhất) trong bài này: giá trị có **tần suất đúng bằng 1** → `{1,4,5,7,8}` — **không có 9, không có 3**.

Nhầm hai khái niệm này là lỗi số 1 của bài.

In [2]:
nums = [5, 7, 3, 9, 4, 9, 8, 3, 1]
print("distinct:", sorted(set(nums)), "-> max =", max(set(nums)))
unique = [x for x in set(nums) if nums.count(x) == 1]
print("unique  :", sorted(unique), "-> max =", max(unique))

distinct: [1, 3, 4, 5, 7, 8, 9] -> max = 9
unique  : [1, 4, 5, 7, 8] -> max = 8


### 2.1. `list.count(x)` — đếm bằng cách quét

`nums.count(x)` duyệt **cả list** để đếm → `O(n)` mỗi lần gọi. Tiện nhưng gọi nhiều lần sẽ chậm.

### 2.2. Đếm tần suất bằng `dict`

Duyệt một lần, mỗi phần tử tăng bộ đếm: `freq[x] = freq.get(x, 0) + 1`. Sau đó tra `freq[x]` trung bình `O(1)`.

In [3]:
freq = {}
for x in [5, 7, 3, 9, 4, 9, 8, 3, 1]:
    freq[x] = freq.get(x, 0) + 1
print(freq)
print("freq[9] =", freq[9], "| freq.get(100, 0) =", freq.get(100, 0))

{5: 1, 7: 1, 3: 2, 9: 2, 4: 1, 8: 1, 1: 1}
freq[9] = 2 | freq.get(100, 0) = 0


### 2.3. `collections.Counter` — dict chuyên đếm

`Counter(nums)` làm đúng việc trên trong một dòng. Truy cập key không có trả `0` (không lỗi). `most_common()` trả danh sách `(giá trị, số lần)` giảm dần theo số lần.

In [4]:
from collections import Counter
cnt = Counter([5, 7, 3, 9, 4, 9, 8, 3, 1])
print(cnt)
print("cnt[9] =", cnt[9], "| cnt[100] =", cnt[100])
print("most_common(2) =", cnt.most_common(2))

Counter({3: 2, 9: 2, 5: 1, 7: 1, 4: 1, 8: 1, 1: 1})
cnt[9] = 2 | cnt[100] = 0
most_common(2) = [(3, 2), (9, 2)]


### 2.4. `max(iterable, default=...)`

`max([])` gây lỗi `ValueError` vì list rỗng không có max. Tham số `default` cho giá trị trả về khi rỗng — rất hợp với yêu cầu "không có thì trả `-1`".

In [5]:
try:
    max([])
except ValueError as e:
    print("Lỗi:", e)
print(max([], default=-1), max([3, 8], default=-1))

Lỗi: max() iterable argument is empty
-1 8


## 3. Cách 1 — Brute Force: đếm từng số bằng `count`

In [6]:
from typing import List

def largest_unique_brute(nums: List[int]) -> int:
    best = -1
    for x in nums:
        if nums.count(x) == 1 and x > best:    # count: O(n)
            best = x
    return best

print(largest_unique_brute([5, 7, 3, 9, 4, 9, 8, 3, 1]))   # 8
print(largest_unique_brute([9, 9, 8, 8]))                  # -1

8
-1


`best = -1` vừa là giá trị khởi tạo, vừa là đáp án khi không tìm thấy — và mọi số hợp lệ `>= 0 > -1` nên sẽ thay được.

### Nút thắt

`n` lần gọi `count`, mỗi lần `O(n)` → `O(n²)`. Với `n = 2000` là `4·10^6` phép so sánh — vẫn qua, nhưng ta đang đếm lại **cùng một giá trị** nhiều lần (mỗi bản sao của 9 lại đếm 9 một lần nữa). Hãy đếm **một lần cho tất cả**.

In [7]:
calls = 0
def counting_count(lst, x):
    global calls
    calls += len(lst)          # mỗi lần count quét len(lst) phần tử
    return lst.count(x)

nums = list(range(2000))
for x in nums:
    counting_count(nums, x)
print("n = 2000 -> số phần tử phải quét:", f"{calls:,}")

n = 2000 -> số phần tử phải quét: 4,000,000


## 4. Cách 2 — Sắp xếp giảm dần rồi quét theo nhóm

Sau khi sắp giảm dần, các giá trị bằng nhau **đứng liền nhau** thành nhóm. Quét từ đầu (số lớn nhất): nhóm đầu tiên có **độ dài 1** chính là đáp án — dừng ngay.

Ví dụ: `[9,9,8,7,5,4,3,3,1]` → nhóm `9,9` (dài 2, bỏ) → nhóm `8` (dài 1) → trả 8.

In [8]:
def largest_unique_sort(nums: List[int]) -> int:
    arr = sorted(nums, reverse=True)
    n = len(arr)
    i = 0
    while i < n:
        j = i
        while j < n and arr[j] == arr[i]:     # j chạy hết nhóm cùng giá trị
            j += 1
        if j - i == 1:                        # nhóm dài 1 -> xuất hiện đúng 1 lần
            return arr[i]
        i = j                                 # nhảy sang nhóm kế
    return -1

print(largest_unique_sort([5, 7, 3, 9, 4, 9, 8, 3, 1]))
print(largest_unique_sort([9, 9, 8, 8]))

8
-1


### 4.1. Trace quét nhóm sau khi sắp xếp

Với `[5,7,3,9,4,9,8,3,1]` → sắp giảm dần `[9,9,8,7,5,4,3,3,1]`:

| i (đầu nhóm) | giá trị | j (sau nhóm) | độ dài `j - i` | quyết định |
|---:|---:|---:|---:|---|
| 0 | 9 | 2 | 2 | bỏ, `i = 2` |
| 2 | 8 | 3 | 1 | **trả 8** |

Chỉ cần quét 3 phần tử là dừng — nhờ duyệt từ lớn xuống nhỏ, nhóm dài 1 đầu tiên chắc chắn là đáp án.

In [9]:
def trace_sort_groups(nums):
    arr = sorted(nums, reverse=True)
    print("sau sắp xếp giảm dần:", arr)
    i, n = 0, len(arr)
    while i < n:
        j = i
        while j < n and arr[j] == arr[i]:
            j += 1
        print(f"nhóm giá trị {arr[i]}: i={i}, j={j}, độ dài={j - i}", "-> TRẢ VỀ" if j - i == 1 else "-> bỏ")
        if j - i == 1:
            return arr[i]
        i = j
    print("hết mảng, không có nhóm dài 1 -> -1")
    return -1

_ = trace_sort_groups([5, 7, 3, 9, 4, 9, 8, 3, 1])
print()
_ = trace_sort_groups([9, 9, 8, 8])

sau sắp xếp giảm dần: [9, 9, 8, 7, 5, 4, 3, 3, 1]
nhóm giá trị 9: i=0, j=2, độ dài=2 -> bỏ
nhóm giá trị 8: i=2, j=3, độ dài=1 -> TRẢ VỀ

sau sắp xếp giảm dần: [9, 9, 8, 8]
nhóm giá trị 9: i=0, j=2, độ dài=2 -> bỏ
nhóm giá trị 8: i=2, j=4, độ dài=2 -> bỏ
hết mảng, không có nhóm dài 1 -> -1


Kỹ thuật "hai chỉ số `i`, `j` quét nhóm" rất hay gặp: `i` là đầu nhóm, `j` chạy tới phần tử đầu tiên khác `arr[i]`; độ dài nhóm là `j - i`. Time `O(n log n)` (sắp xếp) + `O(n)` (quét), Space `O(n)` cho bản sao.

## 5. Insight — đếm một lần, rồi chọn

Tách bài thành 2 bước độc lập:
1. **Đếm:** `freq[x]` cho mọi `x` — một lượt `O(n)`.
2. **Chọn:** trong các `x` có `freq[x] == 1`, lấy lớn nhất (hoặc `-1`).

Vì giá trị chỉ từ `0` đến `1000`, bước "chọn" còn có thể làm bằng cách **duyệt giá trị từ 1000 xuống 0**: gặp số đầu tiên có tần suất 1 là đáp án.

## 6. Cách 3 — `Counter` + `max(..., default=-1)` (lời giải chính)

In [10]:
def largest_unique_counter(nums: List[int]) -> int:
    freq = Counter(nums)
    return max((x for x, c in freq.items() if c == 1), default=-1)

print(largest_unique_counter([5, 7, 3, 9, 4, 9, 8, 3, 1]))
print(largest_unique_counter([9, 9, 8, 8]))

8
-1


Đọc biểu thức: "lấy `max` của các `x` sao cho số lần `c` của nó bằng 1; nếu không có `x` nào thì trả `-1`". `(x for x, c in freq.items() if c == 1)` là generator expression; `freq.items()` cho từng cặp `(giá trị, số lần)`.

## 7. TRACE từng bước

Ví dụ 1: `nums = [5,7,3,9,4,9,8,3,1]`.

**Bước đếm:**

| x | freq sau khi thêm |
|---:|---|
| 5 | {5:1} |
| 7 | {5:1, 7:1} |
| 3 | {…, 3:1} |
| 9 | {…, 9:1} |
| 4 | {…, 4:1} |
| 9 | {…, **9:2**} |
| 8 | {…, 8:1} |
| 3 | {…, **3:2**} |
| 1 | {…, 1:1} |

**Bước chọn:** các giá trị có tần suất 1: `5, 7, 4, 8, 1` → max = **8**.

In [11]:
def trace_largest_unique(nums: List[int]) -> int:
    freq = {}
    print("— Bước 1: đếm —")
    for x in nums:
        freq[x] = freq.get(x, 0) + 1
        print(f"thêm {x:>2} -> freq[{x}] = {freq[x]}")
    print("— Bước 2: chọn —")
    best = -1
    for x, c in freq.items():
        if c == 1:
            note = "unique"
            if x > best:
                best = x
                note += f" -> best = {best}"
        else:
            note = f"lặp {c} lần, bỏ"
        print(f"x={x:>2}, c={c}: {note}")
    print("Kết quả:", best)
    return best

_ = trace_largest_unique([5, 7, 3, 9, 4, 9, 8, 3, 1])

— Bước 1: đếm —
thêm  5 -> freq[5] = 1
thêm  7 -> freq[7] = 1
thêm  3 -> freq[3] = 1
thêm  9 -> freq[9] = 1
thêm  4 -> freq[4] = 1
thêm  9 -> freq[9] = 2
thêm  8 -> freq[8] = 1
thêm  3 -> freq[3] = 2
thêm  1 -> freq[1] = 1
— Bước 2: chọn —
x= 5, c=1: unique -> best = 5
x= 7, c=1: unique -> best = 7
x= 3, c=2: lặp 2 lần, bỏ
x= 9, c=2: lặp 2 lần, bỏ
x= 4, c=1: unique
x= 8, c=1: unique -> best = 8
x= 1, c=1: unique
Kết quả: 8


In [12]:
_ = trace_largest_unique([9, 9, 8, 8])

— Bước 1: đếm —
thêm  9 -> freq[9] = 1
thêm  9 -> freq[9] = 2
thêm  8 -> freq[8] = 1
thêm  8 -> freq[8] = 2
— Bước 2: chọn —
x= 9, c=2: lặp 2 lần, bỏ
x= 8, c=2: lặp 2 lần, bỏ
Kết quả: -1


## 8. Cách 4 — Mảng đếm và duyệt ngược từ 1000

Mảng `count` có 1001 ô (chỉ số = giá trị). Đếm xong, duyệt `v` từ `1000` xuống `0`: số đầu tiên có `count[v] == 1` là lớn nhất → trả ngay. Không cần `max`, không cần sắp xếp.

`range(1000, -1, -1)` sinh `1000, 999, ..., 0`: tham số thứ ba là **bước** `-1` (đi lùi), và điểm dừng `-1` **không** được lấy.

In [13]:
print(list(range(5, -1, -1)))     # [5, 4, 3, 2, 1, 0]

def largest_unique_array(nums: List[int]) -> int:
    count = [0] * 1001
    for x in nums:
        count[x] += 1
    for v in range(1000, -1, -1):
        if count[v] == 1:
            return v
    return -1

print(largest_unique_array([5, 7, 3, 9, 4, 9, 8, 3, 1]), largest_unique_array([9, 9, 8, 8]))

[5, 4, 3, 2, 1, 0]
8 -1


## 9. Code nộp LeetCode

In [14]:
from collections import Counter
from typing import List

class Solution:
    def largestUniqueNumber(self, nums: List[int]) -> int:
        freq = Counter(nums)
        return max((x for x, c in freq.items() if c == 1), default=-1)

sol = Solution()
print(sol.largestUniqueNumber([5, 7, 3, 9, 4, 9, 8, 3, 1]))
print(sol.largestUniqueNumber([9, 9, 8, 8]))

8
-1


Bản mảng đếm (không dùng `Counter`, phù hợp khi phỏng vấn muốn thấy tự cài đặt):

In [15]:
class SolutionArray:
    def largestUniqueNumber(self, nums: List[int]) -> int:
        count = [0] * 1001
        for x in nums:
            count[x] += 1
        for v in range(1000, -1, -1):
            if count[v] == 1:
                return v
        return -1

print(SolutionArray().largestUniqueNumber([5, 7, 3, 9, 4, 9, 8, 3, 1]))

8


## 10. Vì sao đúng? — Độ phức tạp — So sánh

**Tính đúng (Counter):** sau bước đếm, `freq[x]` bằng đúng số lần `x` xuất hiện (mỗi phần tử tăng bộ đếm của chính nó đúng 1 lần). Tập `{x : freq[x] == 1}` đúng là tập các số unique; `max` của tập đó là đáp án; tập rỗng → `default=-1`.

**Tính đúng (mảng đếm duyệt ngược):** duyệt `v` giảm dần nên số unique **đầu tiên** gặp được lớn hơn mọi số unique còn lại; nếu duyệt hết mà không gặp thì không có số unique.

**Tính đúng (sắp xếp):** sau khi sắp giảm dần, mọi bản sao của một giá trị nằm liền nhau trong một nhóm; độ dài nhóm = tần suất. Nhóm dài 1 đầu tiên (tính từ trái) có giá trị lớn nhất trong các nhóm dài 1.

Gọi `n = len(nums)`, `V = 1001`, `d` = số giá trị phân biệt.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute Force (`count`) | `O(n²)` | `O(1)` |
| 2. Sắp xếp + quét nhóm | `O(n log n)` | `O(n)` |
| 3. Counter + max | `O(n)` trung bình | `O(d)` |
| 4. Mảng đếm + duyệt ngược | `O(n + V)` | `O(V)` |

## 11. Bộ test — mẫu, biên, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS {actual}")

cases = [
    ("mẫu 1", [5, 7, 3, 9, 4, 9, 8, 3, 1], 8),
    ("mẫu 2", [9, 9, 8, 8], -1),
    ("1 phần tử", [7], 7),
    ("1 phần tử là 0", [0], 0),
    ("0 là số unique duy nhất", [0, 5, 5], 0),
    ("số lớn nhất xuất hiện 3 lần", [10, 10, 10, 2], 2),
    ("biên 1000", [1000, 999, 999], 1000),
    ("tất cả lặp", [1, 1, 2, 2, 3, 3], -1),
]
for name, nums, exp in cases:
    check(name, Solution().largestUniqueNumber(list(nums)), exp)

mẫu 1                            PASS 8
mẫu 2                            PASS -1
1 phần tử                        PASS 7
1 phần tử là 0                   PASS 0
0 là số unique duy nhất          PASS 0
số lớn nhất xuất hiện 3 lần      PASS 2
biên 1000                        PASS 1000
tất cả lặp                       PASS -1


In [17]:
import random
rng = random.Random(1133)
funcs = [largest_unique_brute, largest_unique_sort, largest_unique_counter, largest_unique_array,
         Solution().largestUniqueNumber, SolutionArray().largestUniqueNumber]
for _ in range(3000):
    nums = [rng.randint(0, 15) for _ in range(rng.randint(1, 20))]
    expected = largest_unique_brute(nums)
    for f in funcs:
        assert f(list(nums)) == expected, (f, nums)
print("3000 bộ ngẫu nhiên: 6 cách khớp Brute Force")

3000 bộ ngẫu nhiên: 6 cách khớp Brute Force


## 12. Lỗi người mới hay gặp

### Lỗi 1: Nhầm "unique" với "distinct" — dùng `set`

In [18]:
def wrong_set(nums):
    return max(set(nums))            # SAI: set chỉ bỏ trùng, 9 vẫn còn

print("sai :", wrong_set([5, 7, 3, 9, 4, 9, 8, 3, 1]), "| đúng:", largest_unique_counter([5, 7, 3, 9, 4, 9, 8, 3, 1]))

sai : 9 | đúng: 8


### Lỗi 2: Chỉ so sánh phần tử liền kề trong mảng **chưa sắp xếp**

In [19]:
def wrong_neighbors(nums):
    best = -1
    for i, x in enumerate(nums):
        left = nums[i - 1] if i > 0 else None
        right = nums[i + 1] if i + 1 < len(nums) else None
        if x != left and x != right:     # SAI: bản sao có thể nằm xa
            best = max(best, x)
    return best

print("sai :", wrong_neighbors([9, 1, 9]), "| đúng:", largest_unique_counter([9, 1, 9]))

sai : 9 | đúng: 1


**Vì sao sai:** kiểm tra hàng xóm chỉ đúng khi các bản sao **liền nhau**, tức là mảng đã sắp xếp. **Sửa:** sắp xếp trước (cách 2) hoặc đếm (cách 3).

### Lỗi 3: Quên trường hợp không có số unique

In [20]:
def wrong_no_default(nums):
    freq = Counter(nums)
    return max(x for x, c in freq.items() if c == 1)   # SAI: rỗng -> ValueError

try:
    wrong_no_default([9, 9, 8, 8])
except ValueError as e:
    print("ValueError:", e, "-> sửa: max(..., default=-1)")

ValueError: max() iterable argument is empty -> sửa: max(..., default=-1)


### Lỗi 4: Khởi tạo `best = 0` thay vì `-1`

In [21]:
def wrong_init_zero(nums):
    freq = Counter(nums)
    best = 0                         # SAI: không phân biệt được "không có" với "đáp án là 0"
    for x, c in freq.items():
        if c == 1:
            best = max(best, x)
    return best

print("sai :", wrong_init_zero([9, 9, 8, 8]), "| đúng:", largest_unique_counter([9, 9, 8, 8]))

sai : 0 | đúng: -1


**Vì sao sai:** khi không có số unique phải trả `-1`, nhưng hàm trả `0`. Vì `0` là giá trị hợp lệ của `nums[i]`, giá trị "không tìm thấy" phải nằm **ngoài** miền giá trị → `-1`.

## 13. Mẹo nhận dạng pattern

- Đề nhắc "xuất hiện **đúng một lần** / **k lần** / nhiều nhất / ít nhất" → **đếm tần suất** (`Counter` / dict / mảng đếm).
- Phân biệt rõ: `set` = bỏ trùng; `Counter` = biết số lần.
- Miền giá trị nhỏ đã biết (`0..1000`) → mảng đếm; muốn **lớn nhất** thì duyệt chỉ số **từ trên xuống** và dừng sớm.
- Cần trả giá trị đặc biệt khi rỗng → `max(..., default=...)` hoặc khởi tạo bằng giá trị ngoài miền.
- Bài cùng họ: #387 First Unique Character in a String, #1748 Sum of Unique Elements, #136 Single Number, #2085 Count Common Words With One Occurrence, #1207 Unique Number of Occurrences.

## 14. Biến thể

### 14.1. #1748 Sum of Unique Elements

In [22]:
def sum_of_unique(nums: List[int]) -> int:
    return sum(x for x, c in Counter(nums).items() if c == 1)

print(sum_of_unique([1, 2, 3, 2]), sum_of_unique([1, 1, 1, 1, 1]))   # 4 0

4 0


### 14.2. #387 First Unique Character — unique **đầu tiên theo vị trí**

Đếm trước, rồi duyệt lại **theo thứ tự gốc** và trả chỉ số đầu tiên có tần suất 1.

In [23]:
def first_uniq_char(s: str) -> int:
    freq = Counter(s)
    for i, ch in enumerate(s):
        if freq[ch] == 1:
            return i
    return -1

print(first_uniq_char("leetcode"), first_uniq_char("loveleetcode"), first_uniq_char("aabb"))   # 0 2 -1

0 2 -1


### 14.3. Số xuất hiện đúng `k` lần lớn nhất

Tổng quát hóa điều kiện `c == 1` thành `c == k`.

In [24]:
def largest_with_freq_k(nums: List[int], k: int) -> int:
    return max((x for x, c in Counter(nums).items() if c == k), default=-1)

print(largest_with_freq_k([5, 7, 3, 9, 4, 9, 8, 3, 1], 2))   # 9
print(largest_with_freq_k([5, 7, 3, 9, 4, 9, 8, 3, 1], 3))   # -1

9
-1


## 15. Bài tập (tăng dần)

**Bài 1 — Tính tay:** `nums = [4, 4, 6, 1, 6, 3, 1, 2]`. Lập bảng tần suất, đáp án là gì?

**Bài 2 — Dự đoán output:** `max(set([3, 3, 2]))`, `Counter([3, 3, 2])[3]`, `max([], default=-1)`, `list(range(3, -1, -1))`.

**Bài 3 — TODO:** viết lời giải bằng dict thủ công (không dùng `Counter`).

In [25]:
def exercise_largest_unique(nums: List[int]) -> int:
    # TODO 1: freq = {}; đếm
    # TODO 2: best = -1; duyệt freq.items(), cập nhật khi c == 1
    pass

# assert exercise_largest_unique([5, 7, 3, 9, 4, 9, 8, 3, 1]) == 8
# assert exercise_largest_unique([9, 9, 8, 8]) == -1

**Bài 4 — Nhỏ nhất:** trả về số **nhỏ nhất** xuất hiện đúng một lần (hoặc `-1`), dùng mảng đếm duyệt **xuôi**.

In [26]:
def exercise_smallest_unique(nums: List[int]) -> int:
    # TODO
    pass

# assert exercise_smallest_unique([5, 7, 3, 9, 4, 9, 8, 3, 1]) == 1
# assert exercise_smallest_unique([2, 2]) == -1

**Bài 5 — Top-k unique:** trả về list `k` số unique lớn nhất, giảm dần (ít hơn `k` thì trả hết).

In [27]:
def exercise_top_k_unique(nums: List[int], k: int) -> List[int]:
    # TODO
    pass

# assert exercise_top_k_unique([5, 7, 3, 9, 4, 9, 8, 3, 1], 3) == [8, 7, 5]
# assert exercise_top_k_unique([9, 9, 1], 3) == [1]

**Bài 6 — Mini challenge (luồng dữ liệu):** thiết kế lớp `UniqueTracker` có `add(x)` và `largest()` trả số unique lớn nhất **hiện tại** (hoặc `-1`), với `0 <= x <= 1000`. Gợi ý: giữ mảng đếm; `largest()` duyệt ngược `O(V)`.

In [28]:
class ExerciseUniqueTracker:
    def __init__(self):
        # TODO
        pass
    def add(self, x: int) -> None:
        # TODO
        pass
    def largest(self) -> int:
        # TODO
        pass

# t = ExerciseUniqueTracker(); t.add(5); t.add(9); assert t.largest() == 9
# t.add(9); assert t.largest() == 5

### 15.1. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đếm từng giá trị | 4,6,1 lặp 2 lần | unique: 3, 2 → **3** |
| 2 | set bỏ trùng | Counter đếm | `3`, `2`, `-1`, `[3, 2, 1, 0]` |
| 3 | mục 2.2 | `freq.get(x, 0) + 1` | `if c == 1 and x > best: best = x` |
| 4 | Giống cách 4 | `range(1001)` | trả `v` đầu tiên có `count[v] == 1` |
| 5 | lọc unique | `sorted(..., reverse=True)` | cắt `[:k]` |
| 6 | `self.count = [0]*1001` | `add`: tăng bộ đếm | `largest`: duyệt `range(1000, -1, -1)` |

## 16. Đáp án tham khảo

- **Bài 1:** `{4:2, 6:2, 1:2, 3:1, 2:1}` → unique `3, 2` → **3**.
- **Bài 2:** `3`, `2`, `-1`, `[3, 2, 1, 0]`.

In [29]:
def answer_largest_unique(nums):
    freq = {}
    for x in nums:
        freq[x] = freq.get(x, 0) + 1
    best = -1
    for x, c in freq.items():
        if c == 1 and x > best:
            best = x
    return best

def answer_smallest_unique(nums):
    count = [0] * 1001
    for x in nums:
        count[x] += 1
    for v in range(1001):
        if count[v] == 1:
            return v
    return -1

def answer_top_k_unique(nums, k):
    freq = Counter(nums)
    return sorted((x for x, c in freq.items() if c == 1), reverse=True)[:k]

class AnswerUniqueTracker:
    def __init__(self):
        self.count = [0] * 1001
    def add(self, x):
        self.count[x] += 1
    def largest(self):
        for v in range(1000, -1, -1):
            if self.count[v] == 1:
                return v
        return -1

assert answer_largest_unique([4, 4, 6, 1, 6, 3, 1, 2]) == 3
assert (max(set([3, 3, 2])), Counter([3, 3, 2])[3], max([], default=-1), list(range(3, -1, -1))) == (3, 2, -1, [3, 2, 1, 0])
assert answer_largest_unique([5, 7, 3, 9, 4, 9, 8, 3, 1]) == 8 and answer_largest_unique([9, 9, 8, 8]) == -1
assert answer_smallest_unique([5, 7, 3, 9, 4, 9, 8, 3, 1]) == 1 and answer_smallest_unique([2, 2]) == -1
assert answer_top_k_unique([5, 7, 3, 9, 4, 9, 8, 3, 1], 3) == [8, 7, 5]
assert answer_top_k_unique([9, 9, 1], 3) == [1]
t = AnswerUniqueTracker(); t.add(5); t.add(9); assert t.largest() == 9
t.add(9); assert t.largest() == 5
t.add(5); assert t.largest() == -1
stream = []
tr = AnswerUniqueTracker()
for _ in range(500):
    x = rng.randint(0, 20)
    stream.append(x); tr.add(x)
    assert tr.largest() == largest_unique_brute(stream)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 17. Template nhớ nhanh

```python
freq = Counter(nums)
return max((x for x, c in freq.items() if c == 1), default=-1)

# Miền giá trị nhỏ: mảng đếm + duyệt từ trên xuống
count = [0] * (MAXV + 1)
for x in nums: count[x] += 1
for v in range(MAXV, -1, -1):
    if count[v] == 1: return v
return -1
```

## 18. Checklist submit

- [ ] "Unique" = tần suất **đúng 1**, không phải `set`?
- [ ] Trả `-1` khi không có số nào (không để `max` của rỗng gây lỗi)?
- [ ] Giá trị "không tìm thấy" khởi tạo `-1`, không phải `0`?
- [ ] Duyệt ngược `range(1000, -1, -1)` có lấy cả `0`?
- [ ] Test `[0]`, `[9,9,8,8]`, số lớn nhất bị lặp?